# 06 — Marts e Views para IA (exercício)

**Objetivo:** praticar agregações de negócio e datasets analíticos para IA.

Execute os notebooks em ordem.

## Onde estamos no pipeline

![Star schema alvo](../docs/images/image.png)

Este é o estado final que vamos atingir ao terminar a sessão: uma fato `fact_report` com chaves estrangeiras para quatro dimensões. Cada notebook contribui com uma camada do caminho até esse esquema.

Posição deste notebook (camada **gold**):

> Fonte → Bronze → Staging → Dim/Fact → Checks → Profiling → **Marts/Views** → Dashboard

Aqui você materializa as perguntas de negócio em `mart_*` e expõe recortes reutilizáveis em `vw_ai_*`. Cada mart é um SELECT com JOIN entre `fact_report` e dimensões; a estrutura do star schema é o que torna esses JOINs simples, rápidos e estáveis.

In [ ]:
# Parametros usados pelo Papermill/Airflow.
run_date = None
project_root = None

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    caminhos_common = [
        candidato / "_common.py",
        candidato / "aula02" / "_common.py",
        candidato / "exercicios" / "_common.py",
        candidato / "sessao-02-data-architecture" / "_common.py",
        candidato / "sessao-02-data-architecture" / "exercicios" / "_common.py",
    ]
    encontrado = next((p for p in caminhos_common if p.exists()), None)
    if encontrado:
        sys.path.insert(0, str(encontrado.parent))
        break
else:
    raise FileNotFoundError("Nao encontrei _common.py a partir do diretorio atual.")

from _common import configurar_paths, conectar_duckdb
paths = configurar_paths(project_root)
globals().update(paths)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Banco DuckDB: {DB_PATH}")


## Exercício
Complete trechos dos marts e views.

In [ ]:
con = conectar_duckdb(DB_PATH)
print("Conexão DuckDB aberta.")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE mart_asset_risk_by_type AS
SELECT
    a.asset_type,
    COUNT(*) AS total_reports,
    COUNT(DISTINCT f.asset_id) AS distinct_assets,
    SUM(CASE WHEN a.max_severity = 'critical' THEN 1 ELSE 0 END) AS critical_cnt,
    SUM(CASE WHEN a.max_severity = 'high' THEN 1 ELSE 0 END) AS high_cnt,
    SUM(CASE WHEN a.max_severity = 'medium' THEN 1 ELSE 0 END) AS medium_cnt,
    SUM(CASE WHEN a.max_severity = 'low' THEN 1 ELSE 0 END) AS low_cnt,
    AVG(CASE a.max_severity
        WHEN 'critical' THEN 4
        WHEN 'high' THEN 3
        WHEN 'medium' THEN 2
        WHEN 'low' THEN 1
        ELSE 0
    END) AS risk_score
FROM fact_report f
LEFT JOIN dim_structured_scope a USING (asset_id)
GROUP BY 1
ORDER BY total_reports DESC;
""")

con.sql("""
CREATE OR REPLACE TABLE mart_team_sla_and_outcomes AS
WITH base AS (
    SELECT
        t.handle AS team_handle,
        f.substate,
        CASE
            WHEN f.created_at IS NOT NULL AND f.disclosed_at IS NOT NULL AND f.disclosed_at >= f.created_at
            THEN DATE_DIFF('day', f.created_at, f.disclosed_at)
        END AS days_to_disclosure
    FROM fact_report f
    LEFT JOIN dim_team t USING (team_id)
)
SELECT
    team_handle,
    COUNT(*) AS reports_total,
    quantile_cont(days_to_disclosure, 0.50) AS p50_days_to_disclosure,
    quantile_cont(days_to_disclosure, 0.90) AS p90_days_to_disclosure,
    AVG(days_to_disclosure) AS avg_days_to_disclosure,
    SUM(CASE WHEN substate = 'resolved' THEN 1 ELSE 0 END) AS resolved_cnt,
    SUM(CASE WHEN substate = 'duplicate' THEN 1 ELSE 0 END) AS duplicate_cnt,
    SUM(CASE WHEN substate = 'informative' THEN 1 ELSE 0 END) AS informative_cnt
FROM base
GROUP BY 1
ORDER BY reports_total DESC;
""")

con.sql("""
CREATE OR REPLACE TABLE mart_reporter_quality AS
SELECT
    r.username,
    r.verified,
    COUNT(*) AS reports_total,
    100.0 * SUM(CASE WHEN f.has_bounty THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0) AS bounty_rate_pct,
    AVG(f.vote_count) AS avg_vote_count,
    100.0 * SUM(CASE WHEN LENGTH(COALESCE(TRIM(f.vulnerability_information), '')) BETWEEN 1 AND 199 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0) AS short_writeup_pct
FROM fact_report f
LEFT JOIN dim_reporter r USING (reporter_id)
GROUP BY 1, 2
ORDER BY reports_total DESC;
""")

con.sql("""
CREATE OR REPLACE VIEW vw_ai_triage_training AS
SELECT
    f.report_id,
    f.title,
    f.vulnerability_information,
    a.asset_type,
    a.max_severity,
    w.name AS weakness_name,
    t.handle AS team_handle,
    f.has_bounty,
    f.vote_count,
    f.created_at,
    f.disclosed_at,
    COALESCE(a.max_severity, 'unknown') AS label_severity,
    COALESCE(NULLIF(f.substate, ''), 'unknown') AS label_outcome
FROM fact_report f
LEFT JOIN dim_structured_scope a USING (asset_id)
LEFT JOIN dim_weakness w USING (weakness_id)
LEFT JOIN dim_team t USING (team_id);
""")

con.sql("""
CREATE OR REPLACE VIEW vw_ai_summary_corpus AS
SELECT
    f.report_id,
    f.title,
    f.vulnerability_information,
    LENGTH(COALESCE(f.vulnerability_information, '')) AS text_chars,
    f.vulnerability_information IS NULL OR LENGTH(TRIM(f.vulnerability_information)) = 0 AS missing_text,
    LENGTH(COALESCE(TRIM(f.vulnerability_information), '')) BETWEEN 1 AND 199 AS thin_text,
    a.asset_type,
    w.name AS weakness_name,
    t.handle AS team_handle
FROM fact_report f
LEFT JOIN dim_structured_scope a USING (asset_id)
LEFT JOIN dim_weakness w USING (weakness_id)
LEFT JOIN dim_team t USING (team_id);
""")

con.sql("""
CREATE OR REPLACE VIEW vw_calendarized_asset_type_trend AS
SELECT
    DATE_TRUNC('month', f.created_at) AS month,
    a.asset_type,
    COUNT(*) AS reports
FROM fact_report f
LEFT JOIN dim_structured_scope a USING (asset_id)
WHERE f.created_at IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2;
""")

con.sql("""
CREATE OR REPLACE TABLE mart_bounty_impact AS
WITH base AS (
    SELECT
        a.asset_type,
        f.has_bounty,
        f.substate,
        CASE
            WHEN f.created_at IS NOT NULL AND f.disclosed_at IS NOT NULL AND f.disclosed_at >= f.created_at
            THEN DATE_DIFF('day', f.created_at, f.disclosed_at)
        END AS days_to_disclosure
    FROM fact_report f
    LEFT JOIN dim_structured_scope a USING (asset_id)
)
SELECT
    asset_type,
    has_bounty,
    COUNT(*) AS total_reports,
    quantile_cont(days_to_disclosure, 0.50) AS p50_days_to_disclosure,
    SUM(CASE WHEN substate = 'duplicate' THEN 1 ELSE 0 END) AS duplicate_cnt,
    SUM(CASE WHEN substate = 'informative' THEN 1 ELSE 0 END) AS informative_cnt
FROM base
GROUP BY 1, 2
ORDER BY asset_type, has_bounty;
""")

con.sql("""
CREATE OR REPLACE VIEW vw_program_health_snapshot AS
SELECT
    t.handle AS team_handle,
    a.asset_type,
    COUNT(*) AS reports
FROM fact_report f
LEFT JOIN dim_team t USING (team_id)
LEFT JOIN dim_structured_scope a USING (asset_id)
GROUP BY 1, 2
ORDER BY reports DESC;
""")

con.sql("""
CREATE OR REPLACE TABLE mart_weakness_trend_qtr AS
SELECT
    DATE_TRUNC('quarter', f.created_at) AS quarter,
    w.name AS weakness_name,
    COUNT(*) AS reports
FROM fact_report f
LEFT JOIN dim_weakness w USING (weakness_id)
WHERE f.created_at IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, reports DESC;
""")

for tabela in [
    "mart_asset_risk_by_type", "mart_team_sla_and_outcomes", "mart_reporter_quality",
    "mart_bounty_impact", "mart_weakness_trend_qtr",
]:
    total = con.sql(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"{tabela}: {total:,} linhas")
